# NPS-25-003

Template: Getting_started.ipynb from hepdata_lib

From hepdata_lib:
The following instructions and examples should get you started to get your analysis into [HEPData](https://hepdata.net) using `hepdata_lib`. Please also refer to the [documentation](http://hepdata-lib.readthedocs.io/). While you can also run `hepdata_lib` on your local computer, you can use the [binder](https://mybinder.org/) or [SWAN](http://swan.cern.ch/) services in the browser. Mind that SWAN is only available for people with a CERN account.

Also useful reference: https://github.com/jalimena/HepData_EXO-23-016/tree/main
See "main" function in createHepData_all.py

## General setup

To make sure things are working and `hepdata_lib` is available, run the following command:

In [ ]:
import hepdata_lib
import numpy as np
from hepdata_lib import Submission, Table, Variable
from __future__ import print_function
print("hepdata_lib version", hepdata_lib.__version__)

## Adding a table/figure

In HEPData, figures and table will both be `Table` objects. 

The first column is the mass of phi_2, the second of phi_1, and the third is the median upper limit.

Let's create the table/figure. First, we need to give it a name, which is usually just the identifier in the paper, i.e. "Figure _". The table also needs a description, which is usually the caption. You also need to describe the location, i.e. where to find it in the publication:

In [ ]:
def make2DLimitTable(tableName, isBDT, fileName, imageName):

    table = Table(tableName)
    if isBDT:
        table.description = "Combined 95% CL observed upper limit on the cross section, using the BDT-based event categorization, as a function of scalar masses."
    else: 
        table.description = "Combined 95% CL observed upper limit on the cross section, using the cut-based event categorization, as a function of scalar masses."
   
    table.location = "Results"
    table.keywords["observables"] = ["SIG"]
    table.keywords["reactions"] = [
        "H -> phi_1 phi_2 -> 2 tau 4 b",
        "H -> phi_1 phi_2 -> 2 tau 2 b"
    ]
    #do I need phrases and "particles"?
    data = np.loadtxt(f"NPS25003_inputs/{fileName}", skiprows=0)

    # Column meaning
    y_vals = data[:, 0]   # FIRST column = y bin centers
    x_vals = data[:, 1]   # SECOND column = x bin centers
    z_vals = data[:, 2]   # bin content

    # Build bin edges from centers
    def make_edges(centers):
        centers = np.unique(centers.astype(float))
        edges = np.zeros(len(centers) + 1)
        edges[1:-1] = 0.5 * (centers[1:] + centers[:-1])
        edges[0] = centers[0] - (edges[1] - centers[0])
        edges[-1] = centers[-1] + (centers[-1] - edges[-2])
        return centers, edges

    y_centers, y_edges = make_edges(y_vals)
    x_centers, x_edges = make_edges(x_vals)

    # Independent variables
    phi2_mass = Variable(
        "phi_2 mass",
        is_independent=True,
        is_binned=True,
        units="GeV"
    )

    phi1_mass = Variable(
        "phi_1 mass",
        is_independent=True,
        is_binned=True,
        units="GeV"
    )

    # Map center -> edge tuple
    y_edges_map = {y: (y_edges[i], y_edges[i+1]) for i, y in enumerate(y_centers)}
    x_edges_map = {x: (x_edges[i], x_edges[i+1]) for i, x in enumerate(x_centers)}

    # Only include bins that exist in your data
    phi2_mass.values = [y_edges_map[y] for y in y_vals]
    phi1_mass.values = [x_edges_map[x] for x in x_vals]

    # Dependent variable
    median_limit = Variable(
        "Median limit",
        is_independent=False,
        is_binned=False,
        units="pb"
    )

    median_limit.values = [float(v) for y,x,v in data] 
    median_limit.add_qualifier("SQRT(S)", "13", "TeV")

    # Add to table
    table.add_variable(phi2_mass)
    table.add_variable(phi1_mass)
    table.add_variable(median_limit)

    table.add_image(f"NPS25003_inputs/{imageName}")
    table.add_additional_resource("Original data file", f"NPS25003_inputs/{fileName}", copy_file=True) #to-do: replace with file and image name


In [ ]:
def make1DLimitTable(tableName, isBDT, isCascade, fileName, imageName):

    table = Table(tableName)
    if isCascade:
        table.description = "B(H -> phi_1 phi_2 -> 2 tau 4b) (%)"
    else: 
        table.description = "B(H -> phi_1 phi_2 -> 2 tau 2 b) (%)"
    table.location = "Results"
    #table.keywords["observables"] = ["SIG"]
    table.keywords["reactions"] = [
        "H -> phi_1 phi_2 -> 2 tau 4 b",
        "H -> phi_1 phi_2 -> 2 tau 2 b"
    ]
    #do I need phrases and "particles"?
    data = np.loadtxt(f"NPS25003_inputs/{fileName}", skiprows=0)

    # Column meaning
    y_vals = data[:, 0]   # FIRST column = y bin centers
    x_vals = data[:, 1]   # SECOND column = x bin centers
    z_vals = data[:, 2]   # bin content

    # Build bin edges from centers
    def make_edges(centers):
        centers = np.unique(centers.astype(float))
        edges = np.zeros(len(centers) + 1)
        edges[1:-1] = 0.5 * (centers[1:] + centers[:-1])
        edges[0] = centers[0] - (edges[1] - centers[0])
        edges[-1] = centers[-1] + (centers[-1] - edges[-2])
        return centers, edges

    y_centers, y_edges = make_edges(y_vals)
    x_centers, x_edges = make_edges(x_vals)

    # Independent variables
    phi2_mass = Variable(
        "phi_2 mass",
        is_independent=True,
        is_binned=True,
        units="GeV"
    )

    phi1_mass = Variable(
        "phi_1 mass",
        is_independent=True,
        is_binned=True,
        units="GeV"
    )

    # Map center -> edge tuple
    y_edges_map = {y: (y_edges[i], y_edges[i+1]) for i, y in enumerate(y_centers)}
    x_edges_map = {x: (x_edges[i], x_edges[i+1]) for i, x in enumerate(x_centers)}

    # Only include bins that exist in your data
    phi2_mass.values = [y_edges_map[y] for y in y_vals]
    phi1_mass.values = [x_edges_map[x] for x in x_vals]

    # Dependent variable
    median_limit = Variable(
        "Median limit",
        is_independent=False,
        is_binned=False,
        units="pb"
    )

    median_limit.values = [float(v) for y,x,v in data] 
    median_limit.add_qualifier("SQRT(S)", "13", "TeV")

    # Add to table
    table.add_variable(phi2_mass)
    table.add_variable(phi1_mass)
    table.add_variable(median_limit)

    table.add_image("NPS25003_inputs/plotLimit_2d_allchannels-2.pdf")
    table.add_additional_resource("Original data file", "NPS25003_inputs/median_limits_allchannels.txt", copy_file=True) #to-do: replace with file and image name


## Main Function

The `Submission` object represents the whole HEPData entry and thus carries the top-level meta data that is equally valid for all the tables and variables you may want to enter. The object is also used to create the physical submission files you will upload to the HEPData web interface.

When using `hepdata_lib` to make an entry, you always need to create a `Submission` object. 

In [ ]:
def main():
    submission = Submission()
    submission.read_abstract("NPS25003_inputs/abstract.txt")

    #ADL
    submission.add_additional_resource("ADL file", "NPS25003_inputs/NPS25003.adl", copy_file=True)

    #Production cross section

    #Pythia configurations

    #Signal model UFO Files

    #Generator Process cards

    #Cut flow tables

    #Data distributions of relevant ML input features

    #Small set of input vectors & ML outputs

    #Signal Efficiencies for simplified models' model points

    #Statistical model
    submission.add_table(make2DLimitTable("BDT_allchannels", True, "median_limits_allchannels.txt", "NPS-25-003/plots/plotLimit_2d_allchannels.pdf"))
    submission.add_table(make2DLimitTable("BDT_emu", True, "v7_limits/bdtbased/median_limits_emu.txt", "NPS-25-003/plots/plotLimit_2d_emu.pdf"))
    submission.add_table(make2DLimitTable("BDT_mutau", True, "v7_limits/bdtbased/median_limits_mutau.txt", "NPS-25-003/plots/plotLimit_2d_mutau.pdf"))
    submission.add_table(make2DLimitTable("BDT_etau", True, "v7_limits/bdtbased/median_limits_etau.txt", "NPS-25-003/plots/plotLimit_2d_etau.pdf"))


    for table in submission.tables:
        table.keywords["cmenergies"] = [13000]
    outdir = "NPS25003_output"
    submission.create_files(outdir, remove_old=True)

In [ ]:
!cat NPS25003_output/submission.yaml

In [ ]:
!ls NPS25003_output

In [ ]:
!ls submission.tar.gz